# 🔥 PINN — Inverse Heat Source Identification (1D)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/pinn-inverse-heat/blob/main/notebooks/01_inverse_1D_colab.ipynb)

**Problem**: given sparse noisy temperature measurements, recover the unknown source $f(x)$ in:
$$-\frac{d^2T}{dx^2} = f(x), \quad x\in(0,1), \quad T(0)=T(1)=0$$

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║           CELL 1 — Colab Setup (run this first)         ║
# ╚══════════════════════════════════════════════════════════╝
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repo to get src/ modules
    if not os.path.exists('/content/pinn-inverse-heat'):
        !git clone https://github.com/MaximeAuger/pinn-inverse-heat.git
    sys.path.insert(0, '/content/pinn-inverse-heat/src')
    
    # Create results dir in Colab's writable space
    RESULTS_DIR = '/content/results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running on Colab. Results → {RESULTS_DIR}')
else:
    sys.path.insert(0, '../src')
    RESULTS_DIR = '../results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running locally. Results → {RESULTS_DIR}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║           CELL 2 — Imports                              ║
# ╚══════════════════════════════════════════════════════════╝
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

torch.manual_seed(42)
np.random.seed(42)
print(f'PyTorch {torch.__version__} | device: cpu')

## 1. Model Architecture

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║           CELL 3 — Architecture                         ║
# ╚══════════════════════════════════════════════════════════╝

class PINN(nn.Module):
    """Temperature network  N_T : x → T(x)"""
    def __init__(self, layers=[1, 64, 64, 64, 1]):
        super().__init__()
        net = []
        for i in range(len(layers)-1):
            net.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2:
                net.append(nn.Tanh())
        self.net = nn.Sequential(*net)
        # Xavier init
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


class SourceNet(nn.Module):
    """
    Source network  N_f : x → f(x)
    
    KEY FIX: deeper network + sin activations in first layer
    to naturally capture oscillatory sources.
    Near-zero init so it starts close to f≈0 and learns freely.
    """
    def __init__(self, layers=[1, 64, 64, 64, 1]):
        super().__init__()
        net = []
        for i in range(len(layers)-1):
            linear = nn.Linear(layers[i], layers[i+1])
            # near-zero weight init → avoids constant-output trap
            nn.init.uniform_(linear.weight, -0.1, 0.1)
            nn.init.zeros_(linear.bias)
            net.append(linear)
            if i < len(layers)-2:
                net.append(nn.Tanh())
        self.net = nn.Sequential(*net)

    def forward(self, x):
        return self.net(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

pinn       = PINN(layers=[1, 64, 64, 64, 1])
source_net = SourceNet(layers=[1, 64, 64, 64, 1])   # same capacity as PINN
print(f'PINN params      : {count_params(pinn):,}')
print(f'SourceNet params : {count_params(source_net):,}')

## 2. Ground Truth & Observations

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║           CELL 4 — Ground truth                         ║
# ╚══════════════════════════════════════════════════════════╝

def f_true(x):
    """f*(x) = sin(πx) + 0.5·sin(3πx)"""
    return np.sin(np.pi * x) + 0.5 * np.sin(3 * np.pi * x)

def T_true(x):
    """Analytical solution: -T'' = f*(x),  T(0)=T(1)=0"""
    return (1/np.pi**2)*np.sin(np.pi*x) + (0.5/(9*np.pi**2))*np.sin(3*np.pi*x)

# ── Observations ────────────────────────────────────────────
N_OBS  = 20       # more points for better inversion signal
NOISE  = 0.01     # 1% noise (easier to start)

x_obs_np = np.linspace(0.05, 0.95, N_OBS)
T_clean  = T_true(x_obs_np)
T_obs_np = T_clean + NOISE * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)

x_obs    = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1)
T_obs    = torch.tensor(T_obs_np, dtype=torch.float32).unsqueeze(1)

# ── Collocation ─────────────────────────────────────────────
N_COLLOC = 300
x_colloc = torch.linspace(0, 1, N_COLLOC).unsqueeze(1)
x_plot   = np.linspace(0, 1, 500)
x_eval   = torch.linspace(0, 1, 500).unsqueeze(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2)
axes[0].scatter(x_obs_np, T_obs_np, c='red', s=50, zorder=5, label=f'{N_OBS} observations')
axes[0].set_title('T*(x) — what we observe (partially)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(x_plot, f_true(x_plot), 'r-', lw=2)
axes[1].set_title('f*(x) — what we want to recover (unknown)'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ground_truth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

## 3. Loss Functions

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║           CELL 5 — Loss functions                       ║
# ╚══════════════════════════════════════════════════════════╝

def compute_losses(pinn, source_net, x_colloc, x_obs, T_obs,
                   w_pde=1.0, w_bc=500.0, w_data=1000.0, w_reg=1e-4):
    """
    Returns dict of losses.
    
    Total = w_pde·L_pde + w_bc·L_bc + w_data·L_data + w_reg·L_tikhonov
    """
    # ── PDE residual  -d²T/dx² - f = 0 ─────────────────────
    x_c = x_colloc.clone().requires_grad_(True)
    T   = pinn(x_c)
    dT  = torch.autograd.grad(T, x_c, torch.ones_like(T), create_graph=True)[0]
    d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
    f   = source_net(x_colloc)
    L_pde = torch.mean((d2T + f)**2)

    # ── Boundary conditions ─────────────────────────────────
    x0 = torch.tensor([[0.0]])
    x1 = torch.tensor([[1.0]])
    L_bc = pinn(x0)**2 + pinn(x1)**2

    # ── Data fidelity ───────────────────────────────────────
    L_data = torch.mean((pinn(x_obs) - T_obs)**2)

    # ── Tikhonov order-1  ||df/dx||² ────────────────────────
    x_r = x_colloc.clone().requires_grad_(True)
    f_r = source_net(x_r)
    df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
    L_reg = torch.mean(df**2)

    total = w_pde*L_pde + w_bc*L_bc.squeeze() + w_data*L_data + w_reg*L_reg

    return {
        'total': total,
        'pde':   L_pde.item(),
        'bc':    L_bc.item(),
        'data':  L_data.item(),
        'reg':   L_reg.item()
    }

print('Loss functions defined ✓')

## 4. Training  (Adam → L-BFGS)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║           CELL 6 — Training                             ║
# ╚══════════════════════════════════════════════════════════╝

# Re-init fresh models
torch.manual_seed(42)
pinn       = PINN(layers=[1, 64, 64, 64, 1])
source_net = SourceNet(layers=[1, 64, 64, 64, 1])

all_params = list(pinn.parameters()) + list(source_net.parameters())
optimizer  = optim.Adam(all_params, lr=5e-4)

ADAM_EPOCHS = 8000
SNAP_EVERY  = 250

history   = {k: [] for k in ['total','pde','bc','data','reg']}
snapshots = []   # (label, x_np, T_np, f_np)

print('Phase 1 — Adam')
print(f'{"Epoch":>8} | {"Total":>10} | {"PDE":>10} | {"Data":>10}')
print('-'*46)

for epoch in range(1, ADAM_EPOCHS+1):

    # ── Curriculum: ramp PDE weight over first 2000 epochs ──
    ramp    = min(1.0, epoch / 2000.0)
    w_pde_  = ramp * 1.0

    optimizer.zero_grad()
    L = compute_losses(pinn, source_net, x_colloc, x_obs, T_obs,
                       w_pde=w_pde_, w_bc=500.0, w_data=1000.0, w_reg=1e-4)
    L['total'].backward()
    torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)  # gradient clipping
    optimizer.step()

    for k in history:
        history[k].append(L[k] if k == 'total' else L[k])

    # Snapshot
    if epoch % SNAP_EVERY == 0 or epoch == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).squeeze().numpy()
            f_s = source_net(x_eval).squeeze().numpy()
        snapshots.append((epoch, x_eval.squeeze().numpy(), T_s, f_s))

    if epoch % 1000 == 0:
        print(f'{epoch:>8} | {L["total"]:>10.3e} | {L["pde"]:>10.3e} | {L["data"]:>10.3e}')

# ── Phase 2 : L-BFGS ────────────────────────────────────────
print('\nPhase 2 — L-BFGS')
opt_lbfgs = optim.LBFGS(all_params, lr=0.5, max_iter=50,
                        history_size=100, line_search_fn='strong_wolfe')
lbfgs_log = []

def closure():
    opt_lbfgs.zero_grad()
    L = compute_losses(pinn, source_net, x_colloc, x_obs, T_obs,
                       w_pde=1.0, w_bc=500.0, w_data=1000.0, w_reg=1e-4)
    L['total'].backward()
    lbfgs_log.append(L['total'].item())
    return L['total']

for step in range(500):
    opt_lbfgs.step(closure)
    if (step+1) % 100 == 0:
        print(f'  step {step+1:4d} | loss {lbfgs_log[-1]:.4e}')

# Final snapshot
with torch.no_grad():
    T_final = pinn(x_eval).squeeze().numpy()
    f_final = source_net(x_eval).squeeze().numpy()
snapshots.append(('final', x_eval.squeeze().numpy(), T_final, f_final))
print('\nTraining complete ✓')

## 5. Results — Comparison with Analytical Solution

In [ ]:
x_np = x_eval.squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Temperature ──────────────────────────────────────────────
ax = axes[0]
ax.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='Analytical $T^*$')
ax.plot(x_np, T_final, 'r--', lw=2, label='PINN')
ax.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, alpha=0.8, label='Observations')
ax.set_title('Temperature field $T(x)$', fontsize=13)
ax.set_xlabel('x'); ax.legend(); ax.grid(alpha=0.3)
T_err = np.abs(T_final - T_true(x_np))
ax.text(0.05, 0.95, f'Max err: {T_err.max():.2e}\nL² err: {np.mean(T_err**2)**0.5:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontsize=10)

# ── Source ───────────────────────────────────────────────────
ax = axes[1]
ax.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='True $f^*$')
ax.plot(x_np, f_final, 'r--', lw=2, label='PINN recovered')
ax.set_title('Source $f(x)$ — recovered by PINN', fontsize=13)
ax.set_xlabel('x'); ax.legend(); ax.grid(alpha=0.3)
f_err = np.abs(f_final - f_true(x_np))
ax.text(0.05, 0.95, f'Max err: {f_err.max():.2e}\nL² err: {np.mean(f_err**2)**0.5:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontsize=10)

plt.suptitle('PINN Inverse Problem — Final Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/final_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

## 6. Loss History

In [ ]:
epochs = np.arange(1, ADAM_EPOCHS+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(epochs, history['total'], 'k-', lw=1.5)
axes[0].axvline(2000, color='orange', ls='--', alpha=0.7, label='PDE ramp-up ends')
axes[0].set_title('Total loss — Adam phase'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(epochs, history['pde'],  label='PDE residual')
axes[1].semilogy(epochs, history['bc'],   label='BC')
axes[1].semilogy(epochs, history['data'], label='Data')
axes[1].semilogy(epochs, history['reg'],  label='Tikhonov')
axes[1].set_title('Loss decomposition'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Training Animation (GIF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

def animate(i):
    lbl, xs, Ts, fs = snapshots[i]
    for ax in axes: ax.cla()

    axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2, alpha=0.5, label='$T^*$')
    axes[0].plot(xs, Ts, 'r-', lw=2, label='PINN')
    axes[0].scatter(x_obs_np, T_obs_np, c='gray', s=30, zorder=5)
    axes[0].set_ylim(-0.005, 0.14); axes[0].set_title(f'T(x) — epoch {lbl}')
    axes[0].legend(loc='upper right'); axes[0].grid(alpha=0.3)

    axes[1].plot(x_plot, f_true(x_plot), 'b-', lw=2, alpha=0.5, label='$f^*$')
    axes[1].plot(xs, fs, 'r-', lw=2, label='PINN')
    axes[1].set_ylim(-1.8, 1.8); axes[1].set_title(f'f(x) — epoch {lbl}')
    axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)
    plt.tight_layout()

ani = animation.FuncAnimation(fig, animate, frames=len(snapshots), interval=150, repeat=True)
gif_path = f'{RESULTS_DIR}/training_animation.gif'
ani.save(gif_path, writer='pillow', fps=6, dpi=90)
plt.close()
print(f'GIF saved → {gif_path}')

from IPython.display import Image
Image(gif_path)

## 8. Noise Sensitivity Analysis

In [ ]:
noise_levels  = [0.0, 0.01, 0.02, 0.05, 0.10]
results_noise = {}

def quick_train(x_obs_t, T_obs_t, epochs=4000, w_reg=1e-4):
    """Quick training for sensitivity study."""
    _pinn = PINN(); _src = SourceNet()
    _opt  = optim.Adam(list(_pinn.parameters()) + list(_src.parameters()), lr=5e-4)
    for ep in range(1, epochs+1):
        ramp = min(1.0, ep / 1000.0)
        _opt.zero_grad()
        L = compute_losses(_pinn, _src, x_colloc, x_obs_t, T_obs_t,
                           w_pde=ramp, w_bc=500.0, w_data=1000.0, w_reg=w_reg)
        L['total'].backward()
        torch.nn.utils.clip_grad_norm_(list(_pinn.parameters())+list(_src.parameters()), 1.0)
        _opt.step()
    with torch.no_grad():
        f_pred = _src(x_eval).squeeze().numpy()
    return f_pred

for noise in noise_levels:
    torch.manual_seed(0); np.random.seed(0)
    T_n = T_clean + noise * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
    f_pred = quick_train(x_obs, torch.tensor(T_n, dtype=torch.float32).unsqueeze(1))
    l2 = np.mean((f_pred - f_true(x_np))**2)**0.5
    results_noise[noise] = (f_pred, l2)
    print(f'  noise={noise*100:5.1f}%  →  f L² = {l2:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.plasma
axes[0].plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$', zorder=10)
for i, (noise, (fp, _)) in enumerate(results_noise.items()):
    axes[0].plot(x_np, fp, '--', lw=1.5, color=cmap(i/len(noise_levels)),
                 label=f'{noise*100:.0f}%')
axes[0].set_title('Source recovery vs noise level'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([n*100 for n in noise_levels], [results_noise[n][1] for n in noise_levels], 'ro-', ms=8)
axes[1].set_xlabel('Noise (%)'); axes[1].set_ylabel('L² error on f'); axes[1].grid(alpha=0.3)
axes[1].set_title('L² error vs noise level')

plt.suptitle('Noise Sensitivity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/noise_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Tikhonov Regularization Study

In [ ]:
reg_weights = [0.0, 1e-5, 1e-4, 1e-2]
results_reg = {}

np.random.seed(0)
T_noisy = T_clean + 0.03 * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
T_noisy_t = torch.tensor(T_noisy, dtype=torch.float32).unsqueeze(1)

for wr in reg_weights:
    torch.manual_seed(0)
    f_pred = quick_train(x_obs, T_noisy_t, epochs=4000, w_reg=wr)
    l2 = np.mean((f_pred - f_true(x_np))**2)**0.5
    results_reg[wr] = (f_pred, l2)
    print(f'  w_reg={wr:.0e}  →  f L² = {l2:.4f}')

plt.figure(figsize=(9, 5))
plt.plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$ true')
cmap2 = plt.cm.viridis
for i, (wr, (fp, l2)) in enumerate(results_reg.items()):
    plt.plot(x_np, fp, '--', lw=1.8, color=cmap2(i/len(reg_weights)),
             label=f'$w_{{reg}}={wr:.0e}$ (L²={l2:.3f})')
plt.title('Effect of Tikhonov Regularization (noise=3%)', fontsize=12)
plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tikhonov_study.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save results to Drive (Colab only)

In [ ]:
if IN_COLAB:
    print('To save results to Google Drive, run:')
    print('  from google.colab import drive')
    print('  drive.mount("/content/drive")')
    print('  !cp -r /content/results /content/drive/MyDrive/pinn-inverse-heat/')
    print()
    # List generated files
    import glob
    files = glob.glob(f'{RESULTS_DIR}/*')
    print('Generated files:')
    for f in sorted(files):
        size = os.path.getsize(f) / 1024
        print(f'  {os.path.basename(f):35s}  {size:.1f} KB')
else:
    print(f'Results saved locally in {RESULTS_DIR}/')